In [5]:
"""
gradcam_metrics.py
==================
Compute three spatial metrics for each Grad-CAM heatmap:
  - CAR: Center Activation Ratio
  - PCD: Peak-to-Center Distance
  - AE : Activation Entropy
"""

import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

CAM_DIR   = "exp3/gradcam_s3_output/cams"
MANIFEST_CSV = "exp3/gradcam_s3_output/selection_manifest.csv" 
OUT_CSV   = "exp3/gradcam_s3_output/cam_metrics.csv"
IMG_SIZE  = 224
CENTER_FRAC = 0.60   # central 60% x 60% region
HSV_LOWER = np.array([0, 30, 50])   # Saturation > 30, Value > 50
HSV_UPPER = np.array([180, 255, 255])

def load_cam(npy_path: str) -> np.ndarray:
    cam = np.load(npy_path)
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    cam = cam / (cam.max() + 1e-8)
    return cam


def center_mask(size: int, frac: float) -> np.ndarray:
    m = np.zeros((size, size), dtype=bool)
    lo = int(size * (1 - frac) / 2)
    hi = size - lo
    m[lo:hi, lo:hi] = True
    return m

def create_leaf_mask(image_bgr: np.ndarray) -> np.ndarray:
    """
    Membuat binary mask jaringan daun menggunakan threshold HSV
    yang identik dengan Leaf-Aware IQA pipeline P1.
    
    Args:
        image_bgr: Citra asli BGR ukuran 224x224
        
    Returns:
        mask: float32 array [0,1] berukuran 224x224
    """
    hsv = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, HSV_LOWER, HSV_UPPER)
    return mask.astype(np.float32) / 255.0

def compute_lar(cam: np.ndarray, leaf_mask: np.ndarray) -> float:
    """
    Menghitung Leaf Attention Ratio (LAR).
    
    LAR = Σ(H(x,y) * M_leaf(x,y)) / Σ(H(x,y))
    
    Nilai mendekati 1.0 = atensi terfokus pada jaringan daun
    Nilai mendekati 0.0 = atensi dominan di background/non-daun
    """
    numerator = np.sum(cam * leaf_mask)
    denominator = np.sum(cam) + 1e-8
    return float(numerator / denominator)

def compute_metrics(cam: np.ndarray, leaf_mask: np.ndarray = None):
    c_mask = center_mask(cam.shape[0], CENTER_FRAC)

    # A. Center Activation Ratio
    car = float(cam[c_mask].sum() / (cam.sum() + 1e-8))

    # B. Peak-to-Center Distance (normalized by image diagonal)
    peak = np.unravel_index(cam.argmax(), cam.shape)
    center = (cam.shape[0] / 2, cam.shape[1] / 2)
    diag = np.sqrt(2) * cam.shape[0]
    pcd = float(np.sqrt((peak[0] - center[0])**2 +
                        (peak[1] - center[1])**2) / diag)

    # C. Activation Entropy (spatial)
    p = cam.flatten()
    p = p / (p.sum() + 1e-8)
    p = p[p > 0]
    ae = float(-np.sum(p * np.log2(p)))

    # D. Leaf Attention Ratio (BARU)
    lar = compute_lar(cam, leaf_mask) if leaf_mask is not None else None

    return car, pcd, ae, lar


def classify_quality(car, pcd, ae):
    """Simple rule-based quality tag (Level-1 only)."""
    good = (car > 0.70) and (pcd < 0.20) and (ae < 3.0)
    poor = (car < 0.50) or (pcd > 0.30) or (ae > 3.5)
    if good:
        return "FOCUSED"
    if poor:
        return "DIFFUSE"
    return "MIXED"


def main():
    rows = []
    # Direktori citra asli NPD (sesuaikan path Anda)
    manifest = pd.read_csv(MANIFEST_CSV) 
    id_to_imgpath = dict(zip(manifest['sample_id'], manifest['path']))
    print(f"Loaded manifest: {len(id_to_imgpath)} samples")

    missing_images = 0
    
    for tag in ["s3", "baseline"]:
        tag_dir = Path(CAM_DIR) / tag
        
        if not tag_dir.exists():
            continue
        npy_files = sorted(tag_dir.glob("*_cam_pred.npy"))
        
        for f in npy_files:
            sample_id = f.name.replace(f"_{tag}_cam_pred.npy", "")
            
            # Load heatmap
            cam = load_cam(str(f))
            
            # Load citra asli untuk membuat leaf mask
            # Sesuaikan format nama file citra asli Anda
            rel_path = id_to_imgpath.get(sample_id, None) 
            print(rel_path)
            img_path = Path(rel_path)
            
            if not img_path.exists():
                print(f"WARNING: Image not found for {sample_id}")
                missing_images += 1
                leaf_mask = np.ones((IMG_SIZE, IMG_SIZE), dtype=np.float32)
            else:
                img_bgr = cv2.imread(str(img_path))
                img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
                leaf_mask = create_leaf_mask(img_bgr)
            
            car, pcd, ae, lar = compute_metrics(cam, leaf_mask)
            
            rows.append(dict(
                sample_id=sample_id,
                model=tag,
                car=round(car, 4),
                pcd=round(pcd, 4),
                ae=round(ae, 4),
                lar=round(lar, 4),       # Kolom baru
                quality=classify_quality(car, pcd, ae),
            ))
    
    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False)

    # Summary per model
    print("\n=== Per-model summary ===")
    for tag in ["baseline", "s3"]:
        sub = df[df["model"] == tag]
        if sub.empty:
            continue
        print(f"\n[{tag.upper()}] n={len(sub)}")
        print(f"  CAR : mean={sub['car'].mean():.3f}  "
              f"median={sub['car'].median():.3f}")
        print(f"  PCD : mean={sub['pcd'].mean():.3f}  "
              f"median={sub['pcd'].median():.3f}")
        print(f"  AE  : mean={sub['ae'].mean():.3f}  "
              f"median={sub['ae'].median():.3f}")
        print(f"  LAR : mean={sub['lar'].mean():.3f}  median={sub['lar'].median():.3f}")
        print(f"  Quality distribution:")
        print(sub["quality"].value_counts().to_string())

    # Paired comparison (same sample_id)
    merged = df[df["model"] == "baseline"].merge(
        df[df["model"] == "s3"], on="sample_id", suffixes=("_bl", "_s3"))
    
    if not merged.empty:
        print(f"\n=== Paired S3 vs Baseline ({len(merged)} matched samples) ===")
        for m in ["car", "pcd", "ae", "lar"]:
            diff = (merged[f"{m}_s3"] - merged[f"{m}_bl"]).mean()
            direction = "improved" if (
                (m == "car" and diff > 0) or
                (m == "lar" and diff > 0) or
                (m in ("pcd", "ae") and diff < 0)
            ) else "regressed"
            print(f"  Δ{m.upper():3s} = {diff:+.4f}  ({direction})")

    print(f"\nSaved to: {OUT_CSV}")


if __name__ == "__main__":
    main()

Loaded manifest: 157 samples
new_pv_local/test/Corn_(maize)___Northern_Leaf_Blight/8f9ac1a8-f91f-4895-ba02-a2ff696fa43e___RS_NLB 3486.JPG
new_pv_local/test/Corn_(maize)___Northern_Leaf_Blight/90dd4436-0614-4f6b-ad3b-b8db56ce4de8___RS_NLB  4673_180deg.JPG
new_pv_local/test/Corn_(maize)___Northern_Leaf_Blight/a0d31f49-895a-4921-afb7-a546f6589433___RS_NLB  4670 copy 2_flipTB.jpg
new_pv_local/test/Corn_(maize)___Northern_Leaf_Blight/c4eb96b2-1345-46d0-a796-161042da21df___RS_NLB 0824 copy.jpg
new_pv_local/test/Corn_(maize)___Northern_Leaf_Blight/d3d7a5a3-ee3f-4654-8e90-8e722d433972___RS_NLB 3553.JPG
new_pv_local/test/Corn_(maize)___Northern_Leaf_Blight/e2d99dbe-c4ff-46d8-8e09-38a19f10d2b5___RS_NLB 3533 copy.jpg
new_pv_local/test/Corn_(maize)___healthy/7ba505f3-b357-4e48-be7b-29ebaf50ef9b___R.S_HL 5526 copy 2_flipLR.jpg
new_pv_local/test/Grape___Black_rot/1f20a91e-f33c-4e38-937d-ee259477211e___FAM_B.Rot 5016_flipLR.JPG
new_pv_local/test/Grape___Black_rot/1f20a91e-f33c-4e38-937d-ee259477211e_

In [3]:
import pandas as pd
from scipy import stats

# 1. Load file cam_metrics.csv yang sudah ada
df = pd.read_csv('exp3/gradcam_s3_output/cam_metrics.csv')

# 2. Hapus duplikasi berdasarkan sample_id dan model
# keep='first' memastikan kita mengambil entri berdasarkan prioritas kategori seleksi awal
df_unique = df.drop_duplicates(subset=['sample_id', 'model'], keep='first')

# 3. Verifikasi jumlah sampel unik 
n_baseline = len(df_unique[df_unique['model'] == 'baseline'])
n_s3 = len(df_unique[df_unique['model'] == 's3'])
print(f"✅ Unique Baseline samples: {n_baseline}")
print(f"✅ Unique S3 samples: {n_s3}")

# 4. Pivot data untuk paired analysis
paired = df_unique.pivot_table(index='sample_id', columns='model', values=['car', 'pcd', 'ae','lar'])

# 5. Hitung Paired t-test
metrics = ['car', 'pcd', 'ae', 'lar']
results = []

for m in metrics:
    s0 = paired[(m, 'baseline')]
    s3 = paired[(m, 's3')]
    
    delta = s3.mean() - s0.mean()
    t_stat, p_val = stats.ttest_rel(s3, s0)
    
    # Hitung 95% Confidence Interval
    sem = stats.sem(s3 - s0)
    ci = stats.t.interval(0.95, len(s0)-1, loc=delta, scale=sem)
    
    results.append({
        'Metric': m.upper(),
        'S0 Mean': round(s0.mean(), 3),
        'S3 Mean': round(s3.mean(), 3),
        'Delta (S3-S0)': round(delta, 3),
        't-statistic': round(t_stat, 3),
        'p-value': f"{p_val:.3f}" if p_val >= 0.001 else "<0.001",
        '95% CI': f"[{ci[0]:.3f}, {ci[1]:.3f}]"
    })

# 6. Tampilkan hasil final untuk Table 5
res_df = pd.DataFrame(results)
print("\n" + "="*80)
print("HASIL FINAL TABLE 5 (n=133 UNIQUE SAMPLES)")
print("="*80)
print(res_df.to_string(index=False))
print("="*80)

✅ Unique Baseline samples: 157
✅ Unique S3 samples: 157

HASIL FINAL TABLE 5 (n=133 UNIQUE SAMPLES)
Metric  S0 Mean  S3 Mean  Delta (S3-S0)  t-statistic p-value           95% CI
   CAR    0.471    0.492          0.020        1.032   0.304  [-0.018, 0.059]
   PCD    0.223    0.225          0.002        0.194   0.846  [-0.020, 0.024]
    AE   14.831   14.610         -0.221       -4.904  <0.001 [-0.310, -0.132]
   LAR    0.654    0.646         -0.008       -0.480   0.632  [-0.041, 0.025]


In [7]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ============================================================
# KONFIGURASI
# ============================================================
INPUT_CSV = 'exp3/gradcam_s3_output/cam_metrics.csv'  # File baru dengan LAR
ALPHA = 0.05

# ============================================================
# 1. LOAD & DEDUPLIKASI
# ============================================================
df = pd.read_csv(INPUT_CSV)
df_unique = df.drop_duplicates(subset=['sample_id', 'model'], keep='first')

n_baseline = len(df_unique[df_unique['model'] == 'baseline'])
n_s3 = len(df_unique[df_unique['model'] == 's3'])
print(f"✅ Unique Baseline samples: {n_baseline}")
print(f"✅ Unique S3 samples: {n_s3}")

if n_baseline != n_s3:
    raise ValueError(f"Sample count mismatch! Baseline={n_baseline}, S3={n_s3}")

# ============================================================
# 2. PIVOT UNTUK PAIRED ANALYSIS
# ============================================================
metrics = ['car', 'pcd', 'ae', 'lar']
paired = df_unique.pivot_table(index='sample_id', columns='model', values=metrics)

# Drop sampel yang tidak memiliki pasangan lengkap
paired = paired.dropna()
n_paired = len(paired)
print(f"✅ Paired samples for analysis: {n_paired}")

# ============================================================
# 3. PAIRED T-TEST + COHEN'S DZ + RAW P-VALUES
# ============================================================
results = []
raw_p_values = []

for m in metrics:
    s0 = paired[(m, 'baseline')]
    s3 = paired[(m, 's3')]
    diff = s3 - s0
    
    delta = diff.mean()
    t_stat, p_val = stats.ttest_rel(s3, s0)
    
    # Cohen's dz untuk paired design
    # dz = mean(diff) / std(diff)
    cohens_dz = delta / diff.std(ddof=1) if diff.std(ddof=1) > 0 else 0.0
    
    # 95% CI of the mean difference
    sem = stats.sem(diff)
    ci = stats.t.interval(0.95, n_paired - 1, loc=delta, scale=sem)
    
    raw_p_values.append(p_val)
    results.append({
        'Metric': m.upper(),
        'B0 Mean': round(s0.mean(), 3),
        'P1 Mean': round(s3.mean(), 3),
        'Delta (P1-B0)': round(delta, 3),
        't-statistic': round(t_stat, 3),
        'Raw p-value': p_val,
        '95% CI': f"[{ci[0]:.3f}, {ci[1]:.3f}]",
        "Cohen's dz": round(cohens_dz, 3)
    })

# ============================================================
# 4. BENJAMINI-HOCHBERG FDR CORRECTION
# ============================================================
reject, adjusted_p, _, _ = multipletests(raw_p_values, alpha=ALPHA, method='fdr_bh')

for i, res in enumerate(results):
    adj_p = adjusted_p[i]
    res['Adjusted p-value'] = f"{adj_p:.3f}" if adj_p >= 0.001 else "<0.001"
    res['Significant'] = '✓' if reject[i] else '✗'
    
    # Format raw p juga
    res['Raw p-value'] = f"{res['Raw p-value']:.3f}" if res['Raw p-value'] >= 0.001 else "<0.001"

# ============================================================
# 5. OUTPUT FINAL TABLE 5
# ============================================================
res_df = pd.DataFrame(results)

# Urutkan kolom sesuai format paper
final_cols = [
    'Metric', 'B0 Mean', 'P1 Mean', 'Delta (P1-B0)',
    't-statistic', 'Raw p-value', 'Adjusted p-value',
    '95% CI', "Cohen's dz", 'Significant'
]
res_df = res_df[final_cols]

print("\n" + "=" * 100)
print(f"HASIL FINAL TABLE 5 — Paired Spatial Metrics (B0 vs P1, n={n_paired})")
print(f"Multiple comparison correction: Benjamini-Hochberg FDR (α={ALPHA})")
print("=" * 100)
print(res_df.to_string(index=False))
print("=" * 100)

# Simpan ke CSV untuk copy-paste ke Word/LaTeX
output_path = 'exp3/gradcam_s3_output/table5_final.csv'
res_df.to_csv(output_path, index=False)
print(f"\n💾 Saved to: {output_path}")

# ============================================================
# 6. INTERPRETASI OTOMATIS
# ============================================================
print("\n📊 INTERPRETASI:")
for _, row in res_df.iterrows():
    sig_text = "SIGNIFIKAN" if row['Significant'] == '✓' else "TIDAK signifikan"
    direction = ""
    m = row['Metric']
    d = float(row["Cohen's dz"])
    
    if m in ('CAR', 'LAR'):
        direction = "meningkat" if d > 0 else "menurun"
    else:  # PCD, AE
        direction = "menurun" if d < 0 else "meningkat"
    
    effect = "small" if abs(d) < 0.2 else "medium" if abs(d) < 0.5 else "large"
    
    print(f"  • {m}: {sig_text} setelah BH correction | "
          f"Cohen's dz = {d:+.3f} ({effect} effect, {direction})")

✅ Unique Baseline samples: 157
✅ Unique S3 samples: 157
✅ Paired samples for analysis: 157

HASIL FINAL TABLE 5 — Paired Spatial Metrics (B0 vs P1, n=157)
Multiple comparison correction: Benjamini-Hochberg FDR (α=0.05)
Metric  B0 Mean  P1 Mean  Delta (P1-B0)  t-statistic Raw p-value Adjusted p-value           95% CI  Cohen's dz Significant
   CAR    0.471    0.492          0.020        1.032       0.304            0.607  [-0.018, 0.059]       0.082           ✗
   PCD    0.223    0.225          0.002        0.194       0.846            0.846  [-0.020, 0.024]       0.015           ✗
    AE   14.831   14.610         -0.221       -4.904      <0.001           <0.001 [-0.310, -0.132]      -0.391           ✓
   LAR    0.654    0.646         -0.008       -0.480       0.632            0.843  [-0.041, 0.025]      -0.038           ✗

💾 Saved to: exp3/gradcam_s3_output/table5_final.csv

📊 INTERPRETASI:
  • CAR: TIDAK signifikan setelah BH correction | Cohen's dz = +0.082 (small effect, meningkat)


In [4]:
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

df = pd.read_csv('exp3/gradcam_s3_output/cam_metrics.csv')
manifest = pd.read_csv('exp3/gradcam_s3_output/selection_manifest.csv')
df = df.merge(manifest[['sample_id', 'ref_code']], on='sample_id', how='left')
df = df.drop_duplicates(subset=['sample_id', 'model'], keep='first')

metrics = ['car', 'pcd', 'ae', 'lar']
rows = []
for cls in ['C8', 'C31', 'C35']:
    sub = df[df['ref_code'] == cls]
    piv = sub.pivot_table(index='sample_id', columns='model', values=metrics).dropna()
    entries, raw_ps = [], []
    for m in metrics:
        s0, s3 = piv[(m, 'baseline')], piv[(m, 's3')]
        t, p = stats.ttest_rel(s3, s0)
        dz = (s3 - s0).mean() / (s3 - s0).std(ddof=1)
        entries.append([cls, m.upper(), round(s0.mean(),3), round(s3.mean(),3),
                        round((s3-s0).mean(),3), round(t,3), round(dz,3)])
        raw_ps.append(p)
    _, adj, _, _ = multipletests(raw_ps, alpha=0.05, method='fdr_bh')  # family = 4 metrik per kelas
    for e, a in zip(entries, adj):
        e.append(f"{a:.3f}" if a >= 0.001 else "<0.001")
        rows.append(e)

out = pd.DataFrame(rows, columns=['Class','Metric','B0','P1','Delta','t','dz','Adj p (BH)'])
print(out.to_string(index=False))

Class Metric     B0     P1  Delta      t     dz Adj p (BH)
   C8    CAR  0.371  0.438  0.067  2.317  0.518      0.064
   C8    PCD  0.290  0.264 -0.027 -0.922 -0.206      0.491
   C8     AE 15.054 14.821 -0.234 -3.381 -0.756      0.013
   C8    LAR  0.782  0.801  0.019  0.339  0.076      0.739
  C31    CAR  0.475  0.587  0.112  1.723  0.385      0.202
  C31    PCD  0.206  0.175 -0.030 -0.883 -0.197      0.388
  C31     AE 14.780 14.455 -0.325 -2.385 -0.533      0.111
  C31    LAR  0.598  0.642  0.043  0.975  0.218      0.388
  C35    CAR  0.414  0.477  0.064  1.059  0.237      0.303
  C35    PCD  0.267  0.210 -0.056 -1.562 -0.349      0.284
  C35     AE 14.930 14.748 -0.182 -1.532 -0.343      0.284
  C35    LAR  0.503  0.556  0.053  1.250  0.279      0.302
